# LambdaG Visualization — In primum caput Ecclesiastes vs. Hildebert (H)

Token-level LambdaG visualization comparing *In primum caput Ecclesiastes* against all of Hildebert's known works. Displays the 100 most dissimilar sentences (lowest λG) with masked POS tags and unmasked Latin text side by side, colour-coded by relative lambdaG score.

In [9]:
import numpy as np
import pandas as pd
from sentence_corpora import TwoLevelCorpus
from lambdag.visualize import lambdaG_visualize

# Load corpora
train_corpus = TwoLevelCorpus.from_pickle("train_corpus.pkl")
test_corpus = TwoLevelCorpus.from_pickle("test_corpus.pkl")
ecclesiastes_corpus = TwoLevelCorpus.from_pickle("ecclesiastes_corpus.pkl")

# Combine train + test for the full known/reference pool (excluding ecclesiastes)
combined_corpus = TwoLevelCorpus(
    list(train_corpus) + list(test_corpus), levels=["author", "work"]
)

print(f"train_corpus: {len(train_corpus)} sentences")
print(f"test_corpus:  {len(test_corpus)} sentences")
print(f"combined_corpus: {len(combined_corpus)} sentences")
print(f"ecclesiastes_corpus: {len(ecclesiastes_corpus)} sentences")
print(
    f"\nAuthors in combined_corpus: {sorted(combined_corpus.get_unique_values('author'))}"
)
print(
    f"\necclesiastes author: {sorted(set(s.metadata['author'] for s in ecclesiastes_corpus))}"
)

train_corpus: 32672 sentences
test_corpus:  13997 sentences
combined_corpus: 46669 sentences
ecclesiastes_corpus: 79 sentences

Authors in combined_corpus: ['AI', 'AeC', 'AeP', 'B', 'BC', 'DC', 'E', 'FB', 'GAp', 'GAr', 'GC', 'GM', 'GP', 'GVS', 'GiP', 'H', 'HA', 'HP', 'IH', 'II', 'M', 'NLC', 'OM', 'OSE', 'RC', 'RT', 'SG', 'U1', 'U10', 'U11', 'U2', 'U3', 'U4', 'U5', 'U6', 'U7', 'U8', 'U9']

ecclesiastes author: ['X2']


In [10]:
# --- Balanced sampling using TwoLevelCorpus.sample_balanced ---
rng = np.random.default_rng(seed=42)

# Known author: H (Hildebert) — use ALL works from combined corpus
known_corpus = combined_corpus.filter_by_level("author", "H")
print(f"Hildebert works in combined_corpus: {known_corpus.get_unique_values('work')}")
for work in known_corpus.get_unique_values("work"):
    work_corpus = known_corpus.filter_by_level("work", work)
    print(
        f"  {work}: {len(work_corpus)} sentences, {sum(len(s.text.split()) for s in work_corpus)} tokens"
    )
print(
    f"Total H sentences: {len(known_corpus)}, total tokens: {sum(len(s.text.split()) for s in known_corpus)}"
)

# Use all of H's sentences as known (no sampling needed — we want the full known set)
known_sentences_raw = list(known_corpus)
known_sentences = [list(s.text.split()) for s in known_sentences_raw]
print(f"\nUsing all {len(known_sentences)} known sentences from Hildebert")

# Reference corpus: 30000 tokens from all OTHER authors in combined corpus
ref_corpus = TwoLevelCorpus(
    [s for s in combined_corpus if s.author != "H"], levels=["author", "work"]
)
ref_sentences_raw, ref_breakdown = ref_corpus.sample_balanced(tokens=30000, rng=rng)
ref_sentences = [list(s.text.split()) for s in ref_sentences_raw]
print(f"\nSampled {len(ref_sentences)} reference sentences")
print(f"Reference breakdown: {ref_breakdown}")

Hildebert works in combined_corpus: ['maria']
  maria: 410 sentences, 6591 tokens
Total H sentences: 410, total tokens: 6591

Using all 410 known sentences from Hildebert

Sampled 1614 reference sentences
Reference breakdown: {'AI': {'anticlaudianus': 12077}, 'AeC': {'medicaminum': 1556, 'viaticus': 217}, 'AeP': {'karolinus': 664}, 'B': {'judas': 232}, 'BC': {'contemptu': 757}, 'DC': {'mathildis': 684}, 'E': {'biblia': 398}, 'FB': {'nuptis': 1076}, 'GAp': {'roberti': 584}, 'GAr': {'philippis': 1068}, 'GC': {'alexandreis': 1087}, 'GM': {'merlini': 308}, 'GP': {'ligurinus': 1213}, 'GVS': {'poetria': 405}, 'GiP': {'historia': 454}, 'HA': {'planctus': 392}, 'HP': {'maiolichinus': 572}, 'IH': {'architrenius': 762}, 'II': {'ylias': 563}, 'M': {'capitulorum': 298, 'maurilii': 58, 'victoris': 41}, 'NLC': {'passio': 340}, 'OM': {'ernestus': 544}, 'OSE': {'doctrina': 251}, 'RC': {'tancredi': 156}, 'RT': {'mauri': 121}, 'SG': {'ecclesiasten': 45}, 'U1': {'canticamariae': 46}, 'U10': {'karolellus'

In [15]:
# --- Prepare questioned sentences from librosegum_corpus ---
# q_sentences: tokenized sentences for the visualization
q_sentences = [s.text.split() for s in ecclesiastes_corpus]
print(f"Questioned sentences (In primum caput Ecclesiastes): {len(q_sentences)}")

# unmasked_sents: original sentences for colour-coding with real words
quesplit_file = "data_processed_ecclesiastes/X2_ecclesiastes_sentences.quesplit.txt"
with open(quesplit_file, "r", encoding="utf-8") as f:
    unmasked_sents = [line.strip().split() for line in f if line.strip()]
print(f"Unmasked sentences loaded from quesplit.txt: {len(unmasked_sents)}")

# Verify lengths match
if len(q_sentences) != len(unmasked_sents):
    raise ValueError(
        f"WARNING: Mismatch! q_sentences: {len(q_sentences)}, unmasked_sents: {len(unmasked_sents)}"
    )

Questioned sentences (In primum caput Ecclesiastes): 79
Unmasked sentences loaded from quesplit.txt: 79


In [17]:
# --- Run LambdaG visualization (masked analysis + unmasked colour-coding) ---
result = lambdaG_visualize(
    q_text=q_sentences,
    k_text=known_sentences,
    ref_text=ref_sentences,
    N=8,
    r=30,
    output="html",
    scale="relative",
    negative=True,
    order_by="importance",
    unmasked_sentences=unmasked_sents,
    rng=rng,
)
print(f"Table shape: {result['table'].shape}")
print(f"Sentences: {result['table']['sentence_id'].nunique()}")

Table shape: (1158, 8)
Sentences: 79


In [18]:
# --- Extract last 100 sentence-rows from colour-coded HTML, reverse order ---
from IPython.display import IFrame
import re, base64

html = result["colourcoded_text"]

# Escape <EOL> so it displays as text instead of being hidden by the browser
html = html.replace("<EOL>", "&#60;EOL&#62;")

# Extract all sentence-row divs (each has masked + unmasked columns)
sentence_rows = re.findall(
    r'<div class="sentence-row">.*?</div>\s*</div>', html, re.DOTALL
)
print(f"Total sentence-rows: {len(sentence_rows)}")

# Last 100 = most negative lambdaG = most dissimilar to Hildebert
last100 = sentence_rows[-100:]
last100.reverse()  # most dissimilar first

# Renumber circles
renumbered = []
for i, row in enumerate(last100, 1):
    row = re.sub(
        r'<span class="sentence-number-circle"[^>]*>\d+</span>',
        '<span class="sentence-number-circle" style="background-color: #2166ac; color: white;">'
        + str(i)
        + "</span>",
        row,
    )
    renumbered.append(row)

output_html = (
    '<!DOCTYPE html><html lang="en"><head><meta charset="UTF-8">'
    '<link href="https://fonts.googleapis.com/css2?family=Roboto+Condensed:wght@300;400;700&display=swap" rel="stylesheet">'
    "<style>"
    "body{font-family:'Roboto Condensed',sans-serif;margin:0;padding:20px;background:#f5f5f5;font-weight:300}"
    "h2{text-align:center;color:#333;font-weight:700}"
    ".subtitle{text-align:center;color:#666;margin-bottom:20px;font-size:14px}"
    ".sentence-row{display:flex;align-items:flex-start;margin-bottom:12px;gap:10px}"
    ".sentence-column{flex:1;line-height:1.6;font-size:15px}"
    ".sentence-number-circle{display:inline-block;width:28px;height:28px;line-height:28px;border-radius:50%;text-align:center;font-size:12px;font-weight:700}"
    ".heading-row{display:flex;align-items:center;margin-bottom:10px;gap:10px}"
    ".heading-spacer{width:38px;flex-shrink:0}"
    ".column-heading{flex:1;text-align:center;color:#555;font-size:1.2em;font-weight:400;border-bottom:2px solid #ddd;padding-bottom:10px}"
    "</style></head><body>"
    "<h2>100 Most Dissimilar Sentences — In primum caput Ecclesiastes vs. Hildebert</h2>"
    "<p class='subtitle'>Token-level LambdaG scores with relative scale (most negative = least like Hildebert's style)</p>"
    '<div class="heading-row">'
    '<div class="heading-spacer"></div>'
    '<div class="column-heading">Masked (POS Tags)</div>'
    '<div class="column-heading">Unmasked (Latin Text)</div>'
    "</div>" + "\n".join(renumbered) + "</body></html>"
)

# Display in an IFrame to avoid CSS conflicts with the notebook
html_bytes = output_html.encode("utf-8")
html_b64 = base64.b64encode(html_bytes).decode("utf-8")
IFrame(f"data:text/html;base64,{html_b64}", width="100%", height=800)

Total sentence-rows: 79
